In [5]:
#!/usr/bin/env python3
import os
import pandas as pd

def reverse_readline(fh, buf_size=8192):
    """
    A generator that returns the lines of a file in reverse order.
    Reads file by blocks from the end. (Works for text files opened in binary mode.)
    """
    segment = None
    offset = 0
    fh.seek(0, os.SEEK_END)
    position = fh.tell() # position where the file was read
    file_size = fh.tell()
    while offset < file_size:
        # Incrementally decrease the position of read
        offset = min(file_size, offset + buf_size)
        position = file_size - offset
        fh.seek(position)

        # Read a buffer size of data
        buffer = fh.read(min(buf_size, offset))

        # split the buffer by newline
        lines = buffer.split(b'\n')

        # The first segment of the current buffer is likely a partial line, so add it to the previous segment.
        if segment is not None:
            if buffer[-1] != ord(b'\n'):
                # If the last char isn't a newline, then the last element is partial.
                lines[-1] += segment
            else:
                lines.append(segment)
        segment = lines[0]

        for line in reversed(lines[1:]):
            line = line.decode('utf-8', errors='replace')
            yield line.strip(), position
    # yield the last remaining segment
    if segment is not None:
        segment = segment.decode('utf-8', errors='replace')
        yield segment.strip(), position

def forward_read_lines(header, date_time, log_file_path, start_offset, num_lines=50):
    """
    Open the file in forward (normal) mode, seek to start_offset, and read num_lines.
    Returns a list of strings (lines).
    """
    lines = []
    with open(log_file_path, "r", encoding="utf-8", errors="replace") as f:
        f.seek(start_offset - 81920) # make sure enough lines are read

        marker_found = False
        count = 0
        while (count < num_lines):
            line = f.readline()

            if header in line and date_time in line:
                marker_found = True
                continue

            if marker_found:
                lines.append(line.rstrip("\n"))
                count = count + 1
    return lines


def parse_table_line(line, header):
    """
    Given a table line starting with <i> <j> <k> ..., parse it into tokens.
    """
    tokens = line.split()

    if header == "Post-reaction cec cation":
        index = ['c','j','icat','cec_cation_vr', 'meq cec_cation_vr', 
                 '-cec_cation_flux_vr * dt', '-cec_cation_flux2_vr * dt', 
                 'background_cec_vr * dt']
    elif header == "Post-reaction cation":
        index = ['c','j','icat','cation_vr','mol cation_vr',
                 'background_flux_vr * dt', 'primary_cation_flux_vr * dt', 
                 'cec_cation_flux_vr * dt', 'cec_cation_flux2_vr * dt', 
                 '-secondary_cation_flux_vr * dt', 
                 '-cation_uptake_vr * dt', 'cation_infl_vr * dt', 
                 '-cation_leached_vr * dt', 'cation_runoff_vr * dt']

    tokens = pd.Series(tokens, index = index).astype(float)

    # You may want to convert tokens to appropriate types, e.g. int or float.
    # For now, we simply return the tokens.
    return tokens


if __name__ == '__main__':
    # Change this to your error log file path
    log_file_path = "/gpfs/wolf2/cades/cli185/proj-shared/ywo/E3SM/output/20250424_UC_Davis_ICB20TRCNPRDCTCBC_3year_rmethod1erw/run/fort.100"

    # Set the problematic grid cell and time step
    latitude = '38.533389999999997' # f'{44.75:.15f}'
    longitude = '-121.76588000000001' # f'{360 - 290.75:.15f}'
    date_time = '1850-01-03_06:00:00'

    # Open the file in binary mode for reverse reading for actual info. 
    collected_lines = {"Post-reaction cec cation": [],
                        "Post-reaction cation": []}  # Will collect lines from diagnostics upward to the key marker
    n_soil_layers = 7 # HBR = 7, elsewhere = 10
    with open(log_file_path, "rb") as fh:
        # Read lines in reverse
        for line, position in reverse_readline(fh):
            filt = latitude in line and longitude in line and date_time in line
            if filt:
                if "Post-reaction cec cation" in line:
                    collected_lines["Post-reaction cec cation"] = forward_read_lines("Post-reaction cec cation", date_time, log_file_path, position, n_soil_layers * 5)
                    continue
                if "Post-reaction cation" in line:
                    collected_lines["Post-reaction cation"] = forward_read_lines("Post-reaction cation", date_time, log_file_path, position, n_soil_layers * 5)
                    break

    # Reverse collected_lines so that they are in original order (from "Post-reaction cec cation" down to diagnostics)
    for key in collected_lines.keys():
        table_lines = collected_lines[key]

        table_lines.reverse()

        for i, line in enumerate(table_lines):
            stripped = line.lstrip()
            if stripped and stripped[0].isdigit():
                table_lines[i] = parse_table_line(line, key)
            else:
                raise Exception("Table line not found after 'Post-reaction cec cation' marker.")

        table_lines = pd.DataFrame(table_lines)
        table_lines['c'] = table_lines['c'].astype(int)
        table_lines['j'] = table_lines['j'].astype(int)
        table_lines['icat'] = table_lines['icat'].astype(int)
        table_lines = table_lines.set_index(['c','j','icat']).sort_index()

        collected_lines[key] = table_lines


In [8]:
collected_lines['Post-reaction cation'].loc[
    collected_lines['Post-reaction cation']['cation_vr'] < 0, :]

,,,cation_vr,mol cation_vr,background_flux_vr * dt,primary_cation_flux_vr * dt,cec_cation_flux_vr * dt,cec_cation_flux2_vr * dt,-secondary_cation_flux_vr * dt,-cation_uptake_vr * dt,cation_infl_vr * dt,-cation_leached_vr * dt,cation_runoff_vr * dt
c,j,icat,,,,,,,,,,,
1,7,5,-0.000058,-6.195298e-09,0.0,0.0,-0.000146,-0.0,-0.0,-0.0,-0.000058,-0.0,-0.0


In [9]:
collected_lines['Post-reaction cec cation'].loc[(1,7,5), :]

cec_cation_vr                108.640433
meq cec_cation_vr              0.834411
-cec_cation_flux_vr * dt       0.000146
-cec_cation_flux2_vr * dt      0.000000
background_cec_vr * dt         0.000000
Name: (1, 7, 5), dtype: float64